In [52]:
import os
import warnings
import json

In [53]:
import torch
import pandas as pd

In [54]:
!pwd

/Users/samalyshev/Projects/analytics/edu/recsys-course-spring-2026/jupyter


In [55]:
from lightning_fabric import seed_everything
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from rectools.dataset import Dataset

In [56]:
from rectools.models import HSTUModel

In [57]:
from rectools import Columns
from rectools.model_selection.splitter import Splitter
from rectools.model_selection import LastNSplitter
from rectools.metrics import (
    CoveredUsers,
    Serendipity,
    NDCG,
    AvgRecPopularity,
    CatalogCoverage,
    Recall,
    SufficientReco,
)

In [58]:
from rectools.models import  SASRecModel
from rectools.model_selection import  cross_validate
from rectools.models.nn.item_net import IdEmbeddingsItemNet
from rectools.models.nn.transformers.utils import  leave_one_out_mask

In [59]:
from utils import  RecallCallback, BestModelLoadCallback, get_results

In [60]:
# Enable deterministic behaviour with CUDA >= 10.2
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FutureWarning)

In [61]:
RANDOM_STATE=42
torch.use_deterministic_algorithms(True)
seed_everything(RANDOM_STATE, workers=True)

Seed set to 42


42

In [62]:
# Prepare trainer function for models

# We use callbacks for calculating recall on validation fold, making model checkpoint based on best recall, early stopping and best model load
# We train for maximum 100 epochs
# This is the most common academic training setup for sequential models

RECALL_K = 10
PATIENCE = 5
DIVERGENCE_TRESHOLD = 0.01
EPOCHS = 10
recall_callback = RecallCallback(k=RECALL_K, progress_bar=True)
# Checkpoints based on best recall
max_recall_ckpt = ModelCheckpoint(
    monitor=f"recall@{RECALL_K}",   # or just pass "val_loss" here,
    mode="max",
    filename="best_recall",
)
early_stopping_recall = EarlyStopping(
    monitor=f"recall@{RECALL_K}",
    mode="max",
    patience=PATIENCE,
    divergence_threshold=DIVERGENCE_TRESHOLD,
)
best_model_load = BestModelLoadCallback("best_recall")
callbacks = [recall_callback, max_recall_ckpt, best_model_load]

# Function to get custom trainer
def get_trainer() -> Trainer:
    return Trainer(
        accelerator="gpu",
        devices=1,
        min_epochs=10,
        max_epochs=EPOCHS,
        deterministic=True,
        enable_model_summary=False,
        enable_progress_bar=True,
        callbacks=callbacks,
        logger = CSVLogger("test_logs"),  # We use CSV logging for this guide but there are many other options
    )

In [63]:
# This splitter will cut off the last interaction for the test
loo_splitter = LastNSplitter(n=1, n_splits=1, filter_cold_users = False, filter_cold_items = False)

# `leave_one_out_mask` passed to the model in the configs below will cut off next to last interaction for validation during training

# Both test splitter and validation mask use stable sorting algorithms,
# As well as RecTools data preparators that generate model training sequences during `fit`

In [64]:
# Prepare test metrics

metrics_add = {}
metrics_recall ={}
metrics_ndcg = {}
k_base =  10
K = [10, 50,100,200]
K_RECS= max(K)
for k in K:
    metrics_recall.update({
            f"recall@{k}": Recall(k=k),
        })
    metrics_ndcg.update({
            f"ndcg@{k}": NDCG(k=k, divide_by_achievable=True),
        })
metrics_add = {
    f"arp@{k_base}": AvgRecPopularity(k=k_base, normalize=True),
    f"coverage@{k_base}": CatalogCoverage(k=k_base, normalize=True),
    f"covered_users@{k_base}": CoveredUsers(k=k_base),
    f"sufficient_reco@{k_base}": SufficientReco(k=k_base),
    f"serendipity@{k_base}": Serendipity(k=k_base),
}
metrics  = metrics_recall | metrics_ndcg | metrics_add
metrics_to_show = ['recall@10', 'ndcg@10', 'recall@50', 'ndcg@50', 'recall@200', 'ndcg@200', 'coverage@10',
                       'serendipity@10']

In [65]:
def evaluate(models: dict, splitter:Splitter,dataset: Dataset, path_to_save_res:str) -> None:
    cv_results = cross_validate(
        dataset=dataset,
        splitter=splitter,
        models=models,
        metrics=metrics,
        k=K_RECS,
        filter_viewed=True,
    )
    cv_results["models_log_dir"] = {}
    for model_name, model in models.items():
        cv_results["models_log_dir"].update({model_name:model.fit_trainer.log_dir})
    with open(path_to_save_res, 'w', encoding='utf-8') as f:
        json.dump(cv_results, f, ensure_ascii=False, indent=4)

In [66]:
config = {
    "session_max_len": 200,
    "lightning_module_kwargs": {"logits_t": 0.05}, # logits scale factor same as in the original repository
    "item_net_block_types": (IdEmbeddingsItemNet,),
    "get_val_mask_func": leave_one_out_mask, # validation mask
    "get_trainer_func": get_trainer,
    "verbose": 1,
    "loss": 'sampled_softmax',
    "n_negatives": 128,
    "use_pos_emb": True,
    "dropout_rate": 0.2,
    "n_factors": 50, # embedding dim
    "n_heads": 1,
    "n_blocks": 2,
    "lr": 0.001,
    "batch_size": 128,
}

# Fitting on tracks data

In [67]:
import glob

In [68]:
DATA_DIR = "/Users/samalyshev/Projects/analytics/edu/recsys-course-spring-2026/jupyter/data_random_recs"

In [69]:
data = pd.concat([
    pd.read_json(data_path, lines=True) 
    for data_path 
    in glob.glob(DATA_DIR + "/*/data.json")
])

In [70]:
data.head()

,message,timestamp,user,track,time,latency,recommendation,experiments
0,next,2026-03-29 22:00:19.363,8164,8410,1.00,0.003377,5923.0,{'I2I': 'C'}
1,next,2026-03-29 22:00:19.363,1570,8971,1.00,0.003816,3238.0,{'I2I': 'C'}
2,next,2026-03-29 22:00:19.366,1570,3238,0.09,0.000562,11221.0,{'I2I': 'C'}
3,next,2026-03-29 22:00:19.368,1570,11221,0.49,0.000510,2004.0,{'I2I': 'C'}
4,next,2026-03-29 22:00:19.370,8164,15380,0.99,0.000471,480.0,{'I2I': 'C'}


In [71]:
data.shape

(41323, 8)

In [72]:
data.rename(columns={"user": "user_id", "track": "item_id", "timestamp": "datetime"}, inplace=True)

In [73]:
data['weight'] = 1.0

In [74]:
dataset_tracks = Dataset.construct(data)

In [75]:
dataset_tracks

Dataset(user_id_map=IdMap(external_ids=array([8164, 1570, 3102, ..., 1366, 8505, 6192])), item_id_map=IdMap(external_ids=array([ 8410,  8971,  3238, ..., 11408,  8208,  3501])), interactions=Interactions(df=       user_id  item_id  weight                datetime
0            0        0     1.0 2026-03-29 22:00:19.363
1            1        1     1.0 2026-03-29 22:00:19.363
2            1        2     1.0 2026-03-29 22:00:19.366
3            1        3     1.0 2026-03-29 22:00:19.368
4            0        4     1.0 2026-03-29 22:00:19.370
...        ...      ...     ...                     ...
41318      521    11864     1.0 2026-03-29 22:01:09.235
41319     3246     1538     1.0 2026-03-29 22:01:09.245
41320     3246     1532     1.0 2026-03-29 22:01:09.251
41321     3246      815     1.0 2026-03-29 22:01:09.256
41322     3246     1589     1.0 2026-03-29 22:01:09.262

[41323 rows x 4 columns]), user_features=None, item_features=None)

In [76]:
hstu_tracks  = HSTUModel(
    relative_time_attention=True,
    relative_pos_attention=True,
    **config
)
models_tracks = {
    "hstu_tracks": hstu_tracks,
}

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [77]:
dataset_name_tracks = "tracks"
pivot_name_tracks = f"pivot_results_{dataset_name_tracks}.json"

In [79]:
evaluate(models_tracks, loo_splitter, dataset_tracks, pivot_name_tracks)

Epoch 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 26/26 [00:05<00:00,  4.47it/s, v_num=7]
Validation: |                                                                                                                                                                                  | 0/? [00:00<?, ?it/s]
Validation: |                                                                                                                                                                                  | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 26/26 [00:05<00:00,  4.55it/s, v_num=7, val_loss=7.200, recall@10=0.00321, train_loss=7.880]
Validation: |                                                                                                                                   

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 26/26 [00:07<00:00,  3.31it/s, v_num=7, val_loss=5.400, recall@10=0.136, train_loss=1.730]


In [80]:
from rectools.dataset.context import get_context

In [81]:
users = sorted(data['user_id'].unique())

In [82]:
users_datetime = data.groupby(['user_id'])['datetime'].max().reset_index()

In [83]:
users_datetime

,user_id,datetime
0,1,2026-03-29 22:00:27.361
1,16,2026-03-29 22:00:54.579
2,17,2026-03-29 22:00:48.391
3,37,2026-03-29 22:00:34.675
4,41,2026-03-29 22:00:26.016
...,...,...
3303,9980,2026-03-29 22:00:36.232
3304,9986,2026-03-29 22:00:43.546
3305,9993,2026-03-29 22:00:57.937
3306,9995,2026-03-29 22:01:03.514


In [84]:
context_all_users = get_context(users_datetime)

In [87]:
%%time
recs_all_users_hstu_tracks_large = hstu_tracks.recommend(
    users, 
    dataset_tracks, 
    k=100, 
    filter_viewed=True, 
    context=context_all_users
)

CPU times: user 3.29 s, sys: 418 ms, total: 3.71 s
Wall time: 1.26 s


In [88]:
recs_all_users_hstu_tracks_large

,user_id,item_id,score,rank
0,1,5444,0.453406,1
1,1,11251,0.342599,2
2,1,73,0.324303,3
3,1,5443,0.315924,4
4,1,6156,0.308847,5
...,...,...,...,...
330795,9996,4305,0.259330,96
330796,9996,11819,0.259044,97
330797,9996,1390,0.258700,98
330798,9996,6636,0.258603,99


In [89]:
recs_dict = recs_all_users_hstu_tracks_large.groupby(["user_id"]).agg({"item_id": list}).to_dict()['item_id']

In [91]:
with open("hstu_recommendations.json", "w") as rf:
    for user, recs in recs_dict.items():
        recommendation = {
            "user": int(user),
            "tracks": recs
        }
        
        rf.write(json.dumps(recommendation) + "\n")